# PharmGuard AI — End-to-end Demo

This notebook walks through the pipeline using the sample dataset. Before running:

```bash
python scripts/ingest_data.py --sample
```


In [ ]:
import sys
sys.path.insert(0, '..')

from src.pipeline import PharmGuardPipeline

pipeline = PharmGuardPipeline.from_config()
print('Pipeline loaded.')

## 1. Single-drug normalization

Brand names, misspellings, and mixed case all resolve to the same canonical generic name.

In [ ]:
for name in ['Lipitor', 'atorvastatin', 'LIPITOR', 'atorvastatn']:
    r = pipeline.normalizer.resolve(name)
    print(f'{name!r:20s} → {r.generic_name}  (method={r.method}, confidence={r.confidence:.0f})')

## 2. The geriatric polypharmacy scenario (from the proposal)

In [ ]:
drugs = ['lisinopril', 'spironolactone', 'metformin', 'atorvastatin',
         'aspirin', 'omeprazole', 'sertraline']

result = pipeline.run(drugs, use_llm=False)

from IPython.display import Markdown
Markdown(result.report)

## 3. Inspect the evidence behind each claim

Every finding in the report traces to a specific record.

In [ ]:
for pair, records in result.retrieval.interactions.items():
    print(f'\n{pair[0]} + {pair[1]}')
    for r in records:
        print(f'  [{r.record_id}] severity={r.severity} condition={r.condition}')

## 4. Honest uncertainty: no-data pairs

Rather than silently omitting pairs with no evidence, PharmGuard surfaces them.

In [ ]:
print(f'{len(result.retrieval.no_data_pairs)} pairs with no data in source:\n')
for a, b in result.retrieval.no_data_pairs[:10]:
    print(f'  {a} + {b}')

## 5. Evaluation — run the metric suite

In [ ]:
from src.data.storage import read_table
from src.config import config
from src.evaluation.metrics import Evaluator
from src.evaluation.test_cases import TEST_CASES
from src.agents.retriever import Retriever
from src.retrieval.interaction_retriever import InteractionRetriever
from src.retrieval.side_effect_retriever import SideEffectRetriever

interactions_df = read_table(config.paths.processed_dir / 'interactions.parquet')
retriever = Retriever(InteractionRetriever().load(), SideEffectRetriever().load())
evaluator = Evaluator(pipeline.normalizer, retriever, interactions_df)

agg = evaluator.evaluate_all(TEST_CASES)
print(f'Cases:     {len(agg.cases)}')
print(f'Recall:    {agg.mean_recall:.3f}')
print(f'Precision: {agg.mean_precision:.3f}')